In [1]:
import pandas as pd

In [2]:
def initialize_training_df():
    training_dataframe = pd.read_csv('Airline_data_training.csv')
    training_dataframe['departure_date'] = pd.to_datetime(training_dataframe['departure_date'])
    training_dataframe['booking_date'] = pd.to_datetime(training_dataframe['booking_date'])
    training_dataframe['Prior_days'] = (training_dataframe['departure_date'] - training_dataframe['booking_date']).dt.days
    training_dataframe['weekday'] = training_dataframe['booking_date'].dt.weekday
    training_dataframe['departure_weekday'] = training_dataframe['departure_date'].dt.weekday

    return training_dataframe

def initialize_validation_df():
    validation_dataframe = pd.read_csv('Airline_data_validation.csv')
    validation_dataframe['departure_date'] = pd.to_datetime(validation_dataframe['departure_date'])
    validation_dataframe['booking_date'] = pd.to_datetime(validation_dataframe['booking_date'])
    validation_dataframe['Prior_days'] = (validation_dataframe['departure_date'] - validation_dataframe['booking_date']).dt.days
    validation_dataframe['weekday'] = validation_dataframe['booking_date'].dt.weekday
    validation_dataframe['departure_weekday'] = validation_dataframe['departure_date'].dt.weekday

    return validation_dataframe

In [3]:
def train_simple_model(training_dataframe):
    avg_cumulative_booking = training_dataframe.groupby('Prior_days')['cum_bookings'].mean().reset_index()
    avg_cumulative_booking.rename(columns={'cum_bookings': 'Avg_cumulative_booking'}, inplace=True)
    avg_final_demand = training_dataframe[training_dataframe['Prior_days'] == 0]['cum_bookings'].mean()
    avg_cumulative_booking['Avg_final_demand'] = avg_final_demand

    avg_cumulative_booking['Avg_remaining_demand'] = (
            avg_cumulative_booking['Avg_final_demand'] - avg_cumulative_booking['Avg_cumulative_booking'])

    simple_model_additive_multiplicative = avg_cumulative_booking[
        ['Prior_days', 'Avg_cumulative_booking', 'Avg_final_demand', 'Avg_remaining_demand']]

    simple_model_additive_multiplicative['Avg_booking_rate'] = simple_model_additive_multiplicative['Avg_cumulative_booking'] / simple_model_additive_multiplicative['Avg_final_demand']

    return simple_model_additive_multiplicative

def generate_predictions_for_simple_model(validation_dataframe, simple_model_df):
    # Merge validation_df with model_df on 'prior_days'
    merged_df = validation_dataframe.merge(simple_model_df, left_on='Prior_days', right_on='Prior_days', how='left')
    # Calculate additive_model_prediction
    merged_df['additive_model_prediction'] = merged_df['cum_bookings'] + merged_df['Avg_remaining_demand']
    # Calculate multiplicative_model_prediction
    merged_df['multiplicative_model_prediction'] = merged_df['cum_bookings'] / merged_df['Avg_booking_rate']
    # Final dataframe with the new columns
    zero_prior_days_df = merged_df[merged_df['Prior_days'] == 0]
    merged_df = merged_df.merge(
        zero_prior_days_df[['departure_date', 'cum_bookings']],
        on='departure_date',
        how='left',
        suffixes=('', '_zero_prior'))

    merged_df = merged_df.rename(columns={'cum_bookings_zero_prior': 'expected_bookings_ct'})

    return merged_df[['departure_date', 'booking_date', 'Prior_days',
                           'cum_bookings', 'Avg_remaining_demand', 'Avg_booking_rate',
                           'naive_fcst', 'additive_model_prediction', 'multiplicative_model_prediction', 'expected_bookings_ct'                        
                           ]]

In [4]:
def train_weekday_sensitive_model(training_dataframe):
    weekday_final_demand = training_dataframe[training_dataframe['Prior_days'] == 0].groupby('weekday')['cum_bookings'].mean()
    weekday_sensitive_model_addition_multiplicative = training_dataframe.groupby(['departure_weekday', 'Prior_days'])['cum_bookings'].mean().reset_index()
    weekday_sensitive_model_addition_multiplicative['weekday_final_demand'] = weekday_sensitive_model_addition_multiplicative['departure_weekday'].map(weekday_final_demand)
    weekday_sensitive_model_addition_multiplicative = weekday_sensitive_model_addition_multiplicative.rename(columns={'cum_bookings': 'avg_cum_bookings'})
    weekday_sensitive_model_addition_multiplicative['remaining_demand'] = weekday_sensitive_model_addition_multiplicative['weekday_final_demand'] - weekday_sensitive_model_addition_multiplicative['avg_cum_bookings']
    weekday_sensitive_model_addition_multiplicative['booking_rate'] = weekday_sensitive_model_addition_multiplicative['avg_cum_bookings'] / weekday_sensitive_model_addition_multiplicative['weekday_final_demand']
    return weekday_sensitive_model_addition_multiplicative

def generate_predictions_for_ws_model(validation_data_df, ws_model_df):
    output_df = pd.merge(
        validation_data_df,
        ws_model_df[['departure_weekday', 'Prior_days', 'remaining_demand', 'booking_rate']],
        how='left',
        on=['departure_weekday', 'Prior_days']
    )
    output_df['additive_model_prediction'] = output_df['cum_bookings'] + output_df['remaining_demand']
    output_df['multiplicative_model_prediction'] = output_df['cum_bookings'] / output_df['booking_rate']
    zero_prior_days_df = output_df[simple_model_result_df['Prior_days'] == 0]
    output_df = output_df.merge(zero_prior_days_df[['departure_date', 'cum_bookings']],
                                                             on='departure_date',
                                                             how='left',
                                                             suffixes=('', '_zero_prior'))

    output_df = output_df.rename(columns={'cum_bookings_zero_prior': 'expected_bookings_ct'})
    return output_df

In [5]:
def get_errors_from_model(model_with_predictions_df):
    additive_model_error = abs(model_with_predictions_df['additive_model_prediction'] - model_with_predictions_df['expected_bookings_ct']).sum()
    naive_model_error = abs(model_with_predictions_df['naive_fcst'] - model_with_predictions_df['expected_bookings_ct']).sum()
    multiplicative_model_error = abs(model_with_predictions_df['multiplicative_model_prediction'] - model_with_predictions_df['expected_bookings_ct']).sum()

    return naive_model_error, additive_model_error, multiplicative_model_error

def calculate_mase(reference_model_abs_error,model_abs_error):
    return model_abs_error / reference_model_abs_error

In [34]:
training_df=initialize_training_df()
training_df

,departure_date,booking_date,cum_bookings,Prior_days,weekday,departure_weekday
0,2012-08-16,2012-06-17,0,60,6,3
1,2012-08-16,2012-06-18,0,59,0,3
2,2012-08-16,2012-06-19,0,58,1,3
3,2012-08-16,2012-06-20,2,57,2,3
4,2012-08-16,2012-06-21,2,56,3,3
...,...,...,...,...,...,...
4692,2012-10-31,2012-10-27,201,4,5,2
4693,2012-10-31,2012-10-28,208,3,6,2
4694,2012-10-31,2012-10-29,220,2,0,2
4695,2012-10-31,2012-10-30,239,1,1,2


In [7]:
simple_model = train_simple_model(training_df)
simple_model

,Prior_days,Avg_cumulative_booking,Avg_final_demand,Avg_remaining_demand,Avg_booking_rate
0,0,432.935065,432.935065,0.000000,1.000000
1,1,392.623377,432.935065,40.311688,0.906887
2,2,370.064935,432.935065,62.870130,0.854782
3,3,353.415584,432.935065,79.519481,0.816325
4,4,338.987013,432.935065,93.948052,0.782997
...,...,...,...,...,...
56,56,6.532468,432.935065,426.402597,0.015089
57,57,5.077922,432.935065,427.857143,0.011729
58,58,3.714286,432.935065,429.220779,0.008579
59,59,2.623377,432.935065,430.311688,0.006060


In [8]:
validation_df = initialize_validation_df()
validation_df

,departure_date,booking_date,cum_bookings,naive_fcst,Prior_days,weekday,departure_weekday
0,2012-11-01,2012-11-01,269,NaN,0,3,3
1,2012-11-01,2012-10-31,232,265.250000,1,2,3
2,2012-11-01,2012-10-30,214,264.250000,2,1,3
3,2012-11-01,2012-10-29,203,265.083333,3,0,3
4,2012-11-01,2012-10-28,185,258.416667,4,6,3
...,...,...,...,...,...,...,...
205,2012-11-14,2012-11-04,339,471.166667,10,6,2
206,2012-11-14,2012-11-03,335,474.000000,11,5,2
207,2012-11-14,2012-11-02,333,481.166667,12,4,2
208,2012-11-14,2012-11-01,309,470.166667,13,3,2


In [36]:
simple_model_result_df = generate_predictions_for_simple_model(validation_df, simple_model)
total_error_naive_model, total_error_additive_model,total_error_multiplicative_model= get_errors_from_model(simple_model_result_df)
mase_additive_model = calculate_mase(total_error_naive_model,total_error_additive_model)
mase_multiplicative_model = calculate_mase(total_error_naive_model,total_error_multiplicative_model)
simple_model_result_df

,departure_date,booking_date,Prior_days,cum_bookings,Avg_remaining_demand,Avg_booking_rate,naive_fcst,additive_model_prediction,multiplicative_model_prediction,expected_bookings_ct
0,2012-11-01,2012-11-01,0,269,0.000000,1.000000,NaN,269.000000,269.000000,269
1,2012-11-01,2012-10-31,1,232,40.311688,0.906887,265.250000,272.311688,255.820058,269
2,2012-11-01,2012-10-30,2,214,62.870130,0.854782,264.250000,276.870130,250.356343,269
3,2012-11-01,2012-10-29,3,203,79.519481,0.816325,265.083333,282.519481,248.675559,269
4,2012-11-01,2012-10-28,4,185,93.948052,0.782997,258.416667,278.948052,236.271550,269
...,...,...,...,...,...,...,...,...,...,...
205,2012-11-14,2012-11-04,10,339,174.285714,0.597432,471.166667,513.285714,567.428399,634
206,2012-11-14,2012-11-03,11,335,181.662338,0.580394,474.000000,516.662338,577.194542,634
207,2012-11-14,2012-11-02,12,333,189.974026,0.561195,481.166667,522.974026,593.376523,634
208,2012-11-14,2012-11-01,13,309,201.129870,0.535427,470.166667,510.129870,577.109306,634


In [10]:
print("Additive model MASE:", mase_additive_model)
print("Multiplicative model MASE: ", mase_multiplicative_model)

Additive model MASE: 0.835228602800882
Multiplicative model MASE:  1.5215787327715726


In [11]:
weekday_sensitive_model = train_weekday_sensitive_model(training_df)
weekday_sensitive_model

,departure_weekday,Prior_days,avg_cum_bookings,weekday_final_demand,remaining_demand,booking_rate
0,0,0,470.181818,470.181818,0.000000,1.000000
1,0,1,426.363636,470.181818,43.818182,0.906806
2,0,2,412.545455,470.181818,57.636364,0.877417
3,0,3,405.090909,470.181818,65.090909,0.861562
4,0,4,376.545455,470.181818,93.636364,0.800851
...,...,...,...,...,...,...
422,6,56,3.727273,296.545455,292.818182,0.012569
423,6,57,3.545455,296.545455,293.000000,0.011956
424,6,58,3.545455,296.545455,293.000000,0.011956
425,6,59,2.363636,296.545455,294.181818,0.007971


In [12]:
ws_model_result_df = generate_predictions_for_ws_model(validation_df, weekday_sensitive_model)
total_error_naive_model_new_model, total_error_additive_model_ws_model,total_error_multiplicative_model_ws_model= get_errors_from_model(ws_model_result_df)


In [13]:
print("Additive WS model MASE:", total_error_additive_model_ws_model / total_error_naive_model_new_model)
print("Multiplicative WS model MASE: ", total_error_multiplicative_model_ws_model / total_error_naive_model_new_model)

Additive WS model MASE: 0.643957324451688
Multiplicative WS model MASE:  1.6342242123425885
